In [ ]:
using StaticArrays
using Plots

struct Mesh{I <: Integer, F <: Real}
    vertices::Vector{SVector{2, F}}
    segments::Vector{SVector{2, I}}
end

function numvertices(Ω)
    return length(Ω.vertices)
end

function getvertices(Ω)
    return Ω.vertices
end

function getsegments(Ω)
   return Ω.segments
end

function numsegments(Ω)
    return length(Ω.segments)
end

In [ ]:
function meshrectangle(w, b, h)
    @assert w > h
    @assert b > h

    # Create a regular grid of points
    x = range(0, stop=w, step=h)
    y = range(0, stop=b, step=h)
    vs = [SVector(x[i], y[j]) for j in 1:length(y), i in 1:length(x)]
    vs = vec(vs)

    # Create line segments along the boundary of the rectangle
    top = [SVector(x[i], b) for i in 1:length(x)]
    bottom = [SVector(x[i], 0) for i in 1:length(x)]
    left = [SVector(0, y[j]) for j in 1:length(y)]
    right = [SVector(w, y[j]) for j in 1:length(y)]
    segments = [top; bottom; left; right]
    ntop = length(top)
    nbottom = length(bottom)
    nleft = length(left)
    nright = length(right)
    s = [SVector(i, i+1) for i in 1:ntop-1]
    s = [s; [SVector(i, i+1) for i in ntop+1:ntop+nbottom-1]]
    #s = [s; [SVector(i, i+1)
    return vs
end

meshrectangle(8,4,2)

In [ ]:
function meshrectangle(w, b, h)

    @assert w > h
    @assert b > h
    
    m = ceil(Int, w/h)
    n = ceil(Int, b/h)
    
    SV = SVector{2,Real}
    SS = SVector{2,Int}
    
    vertices = [SV( i*h, j*h ) for i in 0:m, j in 0:n]
    segments = [SS( i + j*(m+1), (i+1) + j*(m+1) ) for i in 0:m-1, j in 0:n-1]
    append!(segments, [SS( i + j*(m+1), i + (j+1)*(m+1) ) for i in 0:m-1, j in 0:n-1])
    
    return Mesh(vertices, segments)
end


# Funcion ya buenaza para meshear un rectangulo con esquina inferior izquierda en 0,0

In [ ]:
function meshrectangle(w::Real, b::Real, h::Real)
    @assert w > h
    @assert b > h
    
    n_w = round(Int64, w/h)  #segments alongh width of rectangle
    n_b = round(Int64, b/h)  #segments along thickness of rectangle
    
    SV = SVector{2, Real}
    vs = [SV([i*h, 0.0]) for i in 0:n_w]             # bottom edge
    append!(vs, [SV([w, i*h]) for i in 1:n_b])        # right edge
    append!(vs, [SV([i*h, b]) for i in n_w-1:-1:0])   # top edge
    append!(vs, [SV([0.0, i*h]) for i in n_b-1:-1:1]) # left edge
    
    n = length(vs)
    SS = SVector{2, Int}
    s = [SS([i, mod1(i+1,n)]) for i in 1:n]
    return Mesh(vs, s)
    println("vertices: $vs")
    println("segmentos: $s")
end

meshrectangle(8,4,2)

# Funcion buenaza para plotear aquel gran rectángulo

In [ ]:
using Plots

function plotmesh(Ω::Mesh)
    v = getvertices(Ω)
    s = getsegments(Ω)

    x = [vi[1] for vi in v]
    y = [vi[2] for vi in v]

    scatter(x, y, markersize=3)
    for si in s
        plot!([v[si[1]][1], v[si[2]][1]], [v[si[1]][2], v[si[2]][2]], color="black", linewidth=1)
    end
    xlims!(minimum(x)-0.5, maximum(x)+0.5)
    ylims!(minimum(y)-0.5, maximum(y)+0.5)
    xlabel!("x")
    ylabel!("y")
    title!("Mesh")
end

Ω = meshrectangle(8, 4, 2)
plotmesh(Ω)

# Función mamaloncísima ya para plotear así bien cabrón when vertices and segments vectors are given

In [ ]:
using Plots
mesh= meshcircle(0.2, radius=1.0)
function plotsegments(Ω::Mesh{N, F}) where {N<:Integer, F<:Real}
    v = getvertices(Ω)
    s = getsegments(Ω)
    p = plot(aspect_ratio=:equal, xlims=(-1.2, 1.2), ylims=(-1.2, 1.2), legend=false)
    for i in 1:length(s)
        start_vertex = v[s[i][1]]
        end_vertex = v[s[i][2]]
        plot!(p, [start_vertex[1], end_vertex[1]], [start_vertex[2], end_vertex[2]])
        println(end_vertex)
    end
    
    return p
end
vertices = getvertices(mesh)
plotsegments(mesh)
scatter!([v[1] for v in vertices], [v[2] for v in vertices], mc=:red, ms=2)
